# Comparativa de algoritmos de Machine Learning para la detección temprana de diabetes en mujeres

---



La diabetes mellitus tipo II es una enfermedad metabólica crónica.

Según la Organización Mundial de la Salud (OMS), esta enfermedad constituye uno de los principales retos de la salud pública, tanto en términos de su prevalencia como de gasto sanitario.

La detección temprana de la enfermedad es esencial para evitar sus complicaciones graves. Pero los sistemas de cribado actuales resultan insuficientes para la detección precoz y eficiente. En este punto es donde entran en juego los modelos de Machine Learning (ML).

El objetivo del presente trabajo es desarrollar y evaluar un modelo de clasificación supervisada, para la predicción de diabetes mellitus tipo II utilizando el dataset de *Pima Indians Diabetes Database*.

Las preguntas que pretende responder son:



1.   ¿Es posible predecir la presencia de diabetes a partir de variables clínicas básicas con un rendimiento aceptable para su uso como una herramienta de apoyo a la decisión clínica?
2.   ¿Qué algoritmo de aprendizaje automático ofrece la mejor sensibilidad en la detección de casos positivos, minimizando los falsos negativos?


1. ¿Qué variables clínicas presentan mayor capacidad predictiva y dichas variables son coherentes con la evidencia clínica?





### 1. CARGA DE DATOS





In [ ]:
#----------------------LIBRERÍAS----------------------
#Librerías para la manipulación de datos
import pandas as pd
import numpy as np

#Librerías visualización
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate

#Librerías para visualización mejorada en consola
from rich import print
from rich import box
from rich.table import Table
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax
from IPython.display import display

console = Console()

#----------------------CONFIGURACIÓN----------------------
#Configuración para mostras más columnas y filas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)


#----------------------CARGA DEL DATASET----------------------
console.rule("[#00ffff]Carga del dataset[/#00ffff]", style="bright_magenta")
#Acceso archivo diabetes.csv a través de url de github
url="https://raw.githubusercontent.com/yaizabarrera13-lab/TFM-Portafolio/refs/heads/main/diabetes.csv"

#Carga del datset desde GitHub para trabajar siempre con la misma
#versión del archivo
df = pd.read_csv(url)
#Confirmación de carga por pantalla
print("\n✅Archivo CSV cargado exitosamente\n")
display(df.head())

#----------------------INFORMACIÓN GENERAL----------------------
print(f"Dimensiones del dataset: {df.shape}")
print(f"Tipos de datos del dataset:\n{df.dtypes}")

El dataset está compuesto por 768 registros y 9 variables.

Todas las variables son númericas, por lo que hace que sea más fácil su manipulación en Machine Learning. De estas variables, 8 son variables predictoras y 1 es la variable objetivo.

### 2. INSPECCIÓN DE LOS DATOS Y EDA INICIAL

In [ ]:
#----------------------INFORMACIÓN BÁSICA----------------------
#Información básica sobre el DataFrame
console.rule("[#00ffff]Información básica del DataFrame:[/#00ffff]",
             style="bright_magenta")
print(f" - Tamaño: {df.shape}") #Tamaño total del dataframe
print(f" - Número total de pacientes: {df.shape[0]}") #Numero total de filas
print(f" - Número total de variables: {df.shape[1]}\n") #Numero total cols
print(f" - Nombre de las variables:{df.columns.tolist()}\n") #cols con nombres


#----------------------ESTRUCTURA DEL DATASET----------------------
#Obtención de información general de los datos: tipos de datos,
#valores nulos y la memoria usada
console.rule("[#00ffff]Estructura y nulos explícitos de los datos:[/#00ffff]",
             style="bright_magenta")
df.info()
print("\n")


#----------------------DUPLICADOS----------------------
#Comprobación presencia de duplicados
console.rule("[#00ffff]Presencia de duplicados:[/#00ffff]",
             style="bright_magenta")
number_of_duplicates = df.duplicated().sum()
print(f"Número de entradas duplicadas: {number_of_duplicates}\n")


#----------------------ESTADÍSTICAS DESCRIPTIVAS----------------------
#Resumen de las estadísticas de los datos
console.rule("[#00ffff]Estadísticas descriptivas de los datos:[/#00ffff]",
             style="bright_magenta")
display(df.describe().T) #La T es para que transponga la tabla y se lea mejor


#----------------------VALORES NULOS----------------------
#Verificación de la presencia de valores nulos en el datframe
console.rule("[#00ffff]Valores nulos del DataFrame:[/#00ffff]",
             style="bright_magenta")
df_null=df.isnull().sum() #Número de nulos que presenta cada variable (columnas)
for col, nulos in df_null.items():
  print(f"Número de nulos:\n {col}: {nulos} nulos", "\n")

El dataset está compuesto por 768 pacientes y 8 variables predictoras y 1 variable objetivo(Outcome).

Todas las variables son de tipo numéricas.

No se detecta presencia de duplicados en el dataset.

No se detectan valores nulos en ninguna de las variables, sin embargo, variables como: *Glucose*, *BloodPressure*, *SkinThickness*, *Insulin* y *BMI*, presentan valores mínimos de 0, los cuales no son fisiológicamente posibles ya que son incompatibles don la vida. Sugieren que estos valores 0 son *missing values*.

Se observan variables con una alta concentración de valores 0, con resultados de percentil 25% = 0.00, en variables como *SkinThickness* e *Insuline*.

En cuanto a las distribuciones, se detectan variables con distribuciones asimétricas ya que presentan valores de media>mediana (percentil 50%), lo que sugiere un sesgo positivo, desplazado a  la derecha. Estas vaiables son *Pregnancies*, *Insulin* y *Age*.

La variable Outcome es binaria, 0=No diabetes y 1=Diabetes. Presenta un valor del percentil 50% de 0, lo que sugiere un desbalanceo de clases que deberá tenerse en cuenta.

### 3. LIMPIEZA DE DATOS

In [ ]:
#----------------------DETECCIÓN DE VALORES 0----------------------
#Observación de valores=0 en variables fisiológicamente no plausibles
cols_con_ceros_no_plausibles=["Glucose", "BloodPressure", "SkinThickness",
                              "Insulin", "BMI"]
zero_counts=(df[cols_con_ceros_no_plausibles]==0).sum() #conteo total de valores
#0 en cada cols con 0 no plausibles

#Creación de tanla resumen
zero_table=pd.DataFrame({"Variable": zero_counts.index,
                         "Número de valores 0": zero_counts.values,
                         "Porcentaje de valores faltantes (%)":
                          np.round((zero_counts.values/len(df))*100,2)})
#Ordenar de mayor a menor %
zero_table=zero_table.sort_values(by="Porcentaje de valores faltantes (%)",
                                  ascending=False)
display(zero_table)

#Transformación de la tabla en forma de figura para guardarla como imágen
fig, ax=plt.subplots(figsize=(9,4)) #creación figura horizontal
#ax.axis("tight") #adaptación del espacio a la tabla
ax.axis("off") #quitar los ejes (x, y)


#----------------------TABLA DE VALORES 0----------------------
#creación de la tabla
tabla=ax.table(
    cellText=zero_table.values, #extracción de los valores de zero_table
    colLabels=zero_table.columns, #col como encabezados
    cellLoc="center",
    colLoc="center",
    loc="center" #tabla en el centro del eje
)

tabla.auto_set_font_size(False) #Desactiva ajuste automático del tamaño letra
tabla.set_fontsize(11) #Se indica el tamaño letra deseado
tabla.scale(1, 1.4) #escalado del ancho y alto de la tabla, para
#ser más legible y no quede apretada

#Ajuste del estilo
for (fila, col), celda in tabla.get_celld().items(): #Bucle que pasa por cada una
#de las celdas, devolviendo un diccionario con posiciones (fila, col)
    celda.set_edgecolor('black') #bordes de la celda negro
    celda.set_linewidth(1) #grosor del borde

    #Comprobación de si la celda es el encabezado (fila==0)
    if fila == 0:
        celda.set_text_props(weight='bold', color='black') #Texto negrita+negro
        celda.set_facecolor('white') #fondo blanco
    else:
        celda.set_text_props(color='black') #si no es encabezado solo negro
        celda.set_facecolor('white') #fondo blanco

#Ajuste de los anchos de las cols manualmente
anchos = [0.32, 0.3, 0.47] #lista de los anchos de cada col

for col, ancho in enumerate(anchos):
    for fila in range(len(zero_table) + 1):  #+1 para contar el encabezado
    #al recorrer las filas
        tabla[(fila, col)].set_width(ancho) #asigna el ancho a cada fila según
        #la col que se encuentre

plt.savefig("tabla_valores_cero.png", bbox_inches="tight", dpi=300) #Guardado
#de la figura como imágen
plt.show()

En este caso, los **valores faltantes están codificados como 0** en las variables clínicas.

La variable *Insulin* es la que presenta mayor porcentaje (48.7%) seguida de *SkinThickness* (29.56%).

*BloodPressure*, *BMI* y *Glucosa* presentan porcentajes bajos (<5%).

Los valores 0 fisiológicamente no plausibles se consideran *missing values* y han de ser tratados antes del entrenamiento de los modelos.

Para permitir su imputación y que sean detectados como valores faltantes, se han de transformar en NaNs.

In [ ]:
#----------------------CONVERSIÓN DE 0 A NANS----------------------
#Hacemos una copia del df inicial donde se aplica la conversión de 0 a NaNs
df_con_nans=df.copy()

#Conversión de los valores 0 (nulos) de las variables a NaN para poder ser
#detectados e imputados después
cols_con_ceros_no_plausibles=["Glucose", "BloodPressure", "SkinThickness",
                              "Insulin", "BMI"]
df_con_nans[cols_con_ceros_no_plausibles]=df_con_nans[cols_con_ceros_no_plausibles].replace(0, np.nan)

#Copia del df con NaNs antes de imputar
df_antes_imputar=df_con_nans.copy()

#Comprobación de que se han convertido los 0 de las cols con 0 no plausibles
#en NaNs
console.rule("[#00ffff]Comprobación de la conversión a NaNs[/#00ffff]",
             style="bright_magenta")
df_antes_imputar.info()
print(f"Valores nulos: {df_antes_imputar.isna().sum().sum()}")
console.print(df_antes_imputar.isna().sum(), "\n")


Tras convertir los valores 0 no clínicamente plausibles a NaNs, se observa la presencia de 652 valores nulos.

### 4. EDA FINAL
A continuación, se presenta el análisis exploratorio final de los datos. Se realiza tras la conversión de los 0 no plausibles a NaN, y previo a la separación de los conjuntos train y test, dado que se trata de la exploración descriptiva de los datos. De esta manera, no se introduce data leakage.
Pimeramente se determinará la correlación de las variables predictoras con la variable objetivo Outcome, para determinar las variables más relevantes desde un punto de vista clínico y visual.


In [ ]:
#----------------------HEATMAP DE CORRELACIONES----------------------
#Correlaciones de las vairbales
console.rule("[#00ffff]Heatmap de correlaciones[/#00ffff]",
             style="bright_magenta")
plt.figure(figsize=(8, 6)) #Creación de la figura con esas medidas
#Matriz de correlación de Pearson entre variables numéricas
#dentro de las celdas y paleta de colores cálidos
sns.heatmap(df_antes_imputar.corr(), annot=True, cmap="coolwarm")
plt.title("Matriz de correlaciones")
plt.show()

Tal y como se observa en el heatmap, las variables predictoras que presentan mayor correlación con la variable objetivo Outcome son: Glucose (0.49), BMI (0.31) e Insulin (0.3). EStas son las variables con más relevancia en la predicción de diabetes.

In [ ]:
#----------------------BOXPLOTS----------------------
#Determinación de variables potencialmente predictoras y detección de outliers
console.rule("[#00ffff]Boxplots[/#00ffff]", style="bright_magenta")

console.print("\n") #Espacio

#Boxplot de Glucose comparada entre outcome=1 y outcome=0
plt.figure(figsize=(6,4)) #Creación de la figura con esas medidas
sns.boxplot(x="Outcome", y="Glucose", data=df_antes_imputar, color="#66c2a5")
plt.title("Niveles de glucosa según presencia de diabetes")
plt.show()
#Boxplot de BMI
plt.figure(figsize=(6,4)) #Creación de la figura con esas medidas
sns.boxplot(x="Outcome", y="BMI", data=df_antes_imputar, color="#66c2a5")
plt.title("BMI según presencia de diabetes")
plt.show()
#Boxplot de Age
plt.figure(figsize=(6,4)) #Creación de la figura con esas medidas
sns.boxplot(x="Outcome", y="Insulin", data=df_antes_imputar, color="#66c2a5")
plt.title("Insulina según presencia de diabetes")
plt.show()
print("\n") #Espacio

#----------------------SCATTERPLOT----------------------
#Separación de las variables
console.rule("[#00ffff]Gráficos de dispersión[/#00ffff]", style="bright_magenta")

console.print("\n") #Espacio

plt.figure(figsize=(6,4)) #Creación de la figura con esas medidas
#Relación entre glucosa (eje x) y BMI (eje y) coloreadas por las clase Outcome
sns.scatterplot(x="Glucose", y="BMI", hue="Outcome", data=df_antes_imputar,
                color="#66c2a5")
plt.title("Relación entre los niveles de glucosa y el BMI y Outcome")
plt.show()
print("\n") #Espacio


#----------------------COUNTPLOT----------------------
#Countplot para observar el balance de clases
console.rule("[#00ffff]Balance de clases[/#00ffff]",
             style="bright_magenta")
console.print("\n") #Espacio
plt.figure(figsize=(6,4)) #Creación de la figura con esas medidas
ax=sns.countplot(x="Outcome", data=df_antes_imputar, color="#66c2a5") #contará
#los casos de cada clase
#Añadir números encima de cada barra
for p in ax.patches:
  ax.annotate(f"{int(p.get_height())}",
              (p.get_x() + p.get_width() / 2., p.get_height()),
              ha="center", va="bottom")
#Etiquetas
plt.title("Distribución variable objetivo (Outcome)")
plt.xlabel("Outcome (0=No diabetes, 1=Diabetes)")
plt.ylabel("Número de pacientes")
plt.show()
print("\n") #Espacio


#----------------------HISTOGRAMA----------------------
#Histograma de variables
console.rule("[#00ffff]Distribución variables predictoras[/#00ffff]",
             style="bright_magenta")
console.print("\n") #Espacio

cols=df_antes_imputar.columns.drop("Outcome") #variable cols que tenga solo las
#variables predictoras
plt.figure(figsize=(10,8)) #Creación de la figura con esas medidas
plt.suptitle("Distribución variables predictoras", fontsize=14)

for i, col in enumerate(cols, 1): #Recorre la lista de las variables asignandoles
#un número como posición
  plt.subplot(3, 3, i) #dividir la figura en una cuadrícula con 3 filas,
  #3 columnas y la posición=1
  sns.histplot(df_antes_imputar[col], kde=True, color="#66c2a5")
  #selección de 1 col del df y una kde=True, curva suavizada
  #(representando la densidad de probabilidad)
  plt.xlabel(f"{col}")
  plt.ylabel("Frecuencia")

#Ocultar el último subplot vacío ya que la cuadrícula es de 3x3=9
plt.subplot(3, 3, 9).axis("off")
plt.tight_layout() #ajustar el espaciado entre subplots
plt.show()

Las observaciones extraídas de los gráficos son:

- Respecto a los boxplots, se observa que los pacientes con diabetes (Outcome=1) presenta mayores niveles de glucosa, un mayor BMI y más edad.

- Respecto al scatterplot, se evidencia que hay cierto solapamiento de clases, lo que indica que son más difíciles de separar.

- Hay un desbalanceo de clases evidente, con 500 casos de ausencia de enfermedad y 268 de presencia de enfermedad.

- Los histogramas confirman la asimetria predicha a través de la función describe(). Presencia de sesgos positivos, con colas largas hacia la derecha de variables como *Pregnancies*, *Insulin*, *Age* y *DiabetesPedigreeFunction*.

In [ ]:
#----------------------DETECCIÓN OUTLIERS----------------------
#Detección por el método más común (IQR)
#El valor se considerará atípico si está por < Q1-1.5*IQR o por > Q3+1.5*IQR
console.rule("[#00ffff]Valores atipicos (outliers):[/#00ffff]",
             style="bright_magenta")
print("\n") #Espacio
#Se cogen todas las cols menos Outcome
cols=df_antes_imputar.columns.drop("Outcome")

contenido=[] #lista vacía donde se van a guardar los resultados del bucle

#Resumen de la presencia de outliers por variable
for col in cols:
  valores_validos=df_antes_imputar[col].dropna() #Eliminación de los nan ya que en pandas,
  #el filtro booleano los detecta como False en comparaciones y se descartan
  Q1=valores_validos.quantile(0.25) #Calculo del primer cuartil
  Q3=valores_validos.quantile(0.75) #Calculo del tercer cuartil
  IQR=Q3-Q1 #Calculo del rango intercuartílico
  condicion=(valores_validos < Q1-1.5*IQR) | (valores_validos > Q3+1.5*IQR)
  outliers=valores_validos[condicion] #Filtrado de los valores atípicos
  #Resultado: dataframe outliers con los valores atípicos de cada una de las
  #col numericas que hemos definido
  print(f"{col}: {len(outliers)} valores atípicos",
        "\n", f"Porcentaje: {len(outliers)/len(valores_validos)*100:.4g}%")
  #Cantidad de outliers por columns y su porcentaje respecto al df
  print(outliers.values, "\n")
  #Guardar resumen de cada columna
  contenido.append({
      "Variable": col,
      "Total Valores Atípicos": len(outliers),
      "Porcentaje Valores Atípicos (%)":
      np.round((len(outliers)/len(valores_validos))*100,2)})


#----------------------TABLA DE OUTLIERS----------------------
#Construcción de una tabla con los resultados contenidos en "contenido" fuera
#del bucle
outliers_table=pd.DataFrame(contenido)
#Ordenar de mayor a menor
outliers_table=outliers_table.sort_values(by="Porcentaje Valores Atípicos (%)",
                                  ascending=False)
outliers_table
#Transformación de la tabla en forma de figura para guardarla como imágen
import matplotlib.pyplot as plt
fig, ax=plt.subplots(figsize=(9,4)) #creación figura horizontal
#ax.axis("tight") #adaptación del espacio a la tabla
ax.axis("off") #quitar los ejes (x, y)

#creación de la tabla
tabla=ax.table(
    cellText=outliers_table.values, #extracción de los valores de zero_table
    colLabels=outliers_table.columns, #col como encabezados
    cellLoc="center",
    colLoc="center",
    loc="center" #tabla en el centro del eje
)

tabla.auto_set_font_size(False) #Desactiva ajuste automático del tamaño letra
tabla.set_fontsize(11) #Se indica el tamaño letra deseado
tabla.scale(1, 1.4) #escalado del ancho y alto de la tabla, para
#ser más legible y no quede apretada

#Ajuste del estilo
for (fila, col), celda in tabla.get_celld().items(): #Bucle que pasa por cada una
#de las celdas, devolviendo un diccionario con posiciones (fila, col)
    celda.set_edgecolor('black') #bordes de la celda negro
    celda.set_linewidth(1) #grosor del borde

    #Comprobación de si la celda es el encabezado (fila==0)
    if fila == 0:
        celda.set_text_props(weight='bold', color='black') #Texto negrita+negro
        celda.set_facecolor('white') #fondo blanco
    else:
        celda.set_text_props(color='black') #si no es encabezado solo negro
        celda.set_facecolor('white') #fondo blanco

#Ajuste de los anchos de las cols manualmente
anchos = [0.32, 0.3, 0.47] #lista de los anchos de cada col

for col, ancho in enumerate(anchos):
    for fila in range(len(outliers_table) + 1):  #+1 para contar el encabezado
    #al recorrer las filas
        tabla[(fila, col)].set_width(ancho) #asigna el ancho a cada fila según
        #la col que se encuentre

plt.savefig("tabla_valores_atípicos.png", bbox_inches="tight", dpi=300) #Guardado
#de la figura como imágen
plt.show()

Se detectan valores atípicos en todas las variables predictoras menos en "*Glucose*".

*Insulin* y *DiabetesPedigreeFunction* son las variables con más presencia de valores atípicos.

Los outliers no se eliminan por 2 razones principales:

1) Por razones clínicas, ya que pueden corresponder a valores reales con posible relevancia clínica, como indicadores de obesidad mórbida o de resistencia severa a la insulina. Su eliminación causaría sesgos en el modelo hacia casos moderados, reduciendo su capacidad de detección en pacientes más graves.

2) Por razones metodológicas, los outliers suelen coincidir con valores extremos de glucosa o insulina, y con casos positivos más evidentes. Eliminarlos afectaria a la capacidad del modelo para la detección de casos graves.


### 5. SEPARACIÓN DEL DATASET EN TRAIN Y TEST

In [ ]:
#----------------------LIBRERÍAS----------------------
#Carga de libreria para la separación de los datos en train y test
from sklearn.model_selection import train_test_split

#----------------------SEPARACIÓN X E Y----------------------
#Separación del df df_antes_imputar en dos, un df sin la variable objetivo, y
#otro df que sea solo la variable objetivo outcome
x=df_antes_imputar.drop(columns="Outcome")
y=df_antes_imputar["Outcome"]


#----------------------SEPARACIÓN EN TRAIN Y TEST----------------------
#Separación de la submuestra en 80% para entrenamiento y 20% para test
x_train, x_test, y_train, y_test = train_test_split(x, #df con v.predictoras
                                                    y, #col Outcome
                                                    test_size=0.2, #repartición
                                                    #en 80% train, 20% test
                                                    random_state=42, #semilla fija
                                                    #para reproducibilidad
                                                    stratify=y) #para mantener
#la estratificación de clases en los dos subconjuntos ya que el dataset
#está desbalanceado


#----------------------VISUALIZACIÓN----------------------
console.rule("[#00ffff]Grupos de entrenamiento y test:[/#00ffff]",
             style="bright_magenta")

#Impresión del numero de filas y columnas del X_train con las 5 primeras filas
print(f"Tamaño del conjunto de entrenamiento: {x_train.shape}")
print(x_train.head(), "\n")

#Impresión del numero de filas y columnas del X_test con las 5 primeras filas
print(f"Tamaño del conjunto de test: {x_test.shape}")
print(x_test.head(), "\n")

console.rule("[#00ffff]Distribuciones de las clases:[/#00ffff]",
             style="bright_magenta")
print(f"Distribución de clases en entrenamiento:\n"
f"{y_train.value_counts(normalize=True).round(3)}\n")
print(f"Distribución de clases en test:\n{y_test.value_counts(normalize=True).round(3)}")

El data set está dividido en un 80% entrenamiento y un 20% para test.

Gracias a stratify=y se mantienen las proporciones de clases casi idénticas en ambos conjuntos.

Las distribuciones en entrenamiento y test son muy similares para la variable objetivo, cosa que ayuda a una evaluación representativa del modelo.

Se fija semilla, random_state=42 para asegurar la reproducibilidad de los resultados.




### 6. IMPUTACIÓN
Los datos faltantes deben eliminarse o imputarse, en este caso al ser variables relevantes para el modelo se realizará la imputación.

Se realiza la imputación después de la división del dataset en entrenamiento y *test* para evitar el *data leakage*. Si se hiciera la imputación en el dataset completo, se introduciria información del conjunto de test en el de entrenamiento. En consecuencia, los resultados de las métricas de evaluación se verían afectados, siendo más altos y poco realistas.

Para variables con tasa baja de *missing values* (<5%), *BloodPressure*, *BMI*, y *Glucose*, se aplica imputación simple mediante la mediana, adecuado cuando la proporción de datos faltantes es baja. Resulta más robusta que la media en caso de valores atípicos, preservando mejor la distribución general de la variable.

Para variables con tasa alta de *missing values* (>5%), *SkinThickness* e *Insulin*, superando el 20%, se aplica imputación mediante KNN con k=5. Utiliza la similitud entre observaciones para estimar los valores ausentes.

Aunque la imputación se realiza sobre x_train, x_test también necesita un paso de imputación, pero con los parámetros de train. Esto es para poder evaluar el modelo en condiciones realistas.

Cuando el modelo se despliega en producción, los nuevos pacientes pueden presentar valores faltantes, que han de pasar por el mismo proceso de pre-procesamiento. Para poder realizar las predicciones y calcular las métricas, el conjunto de test debe imputarse, para observar como se comportaría en un escenario con pacientes reales.

In [ ]:
#----------------------LIBRERÍAS-----------------------
#Carga de libreria necesaria
from sklearn.impute import SimpleImputer, KNNImputer

#Copias de los conjuntas para trabajar con ellas
x_train_imputed=x_train.copy()
x_test_imputed=x_test.copy()


#----------------------IMPUTACIÓN SIMPLE-----------------------
#Imputación simple (mediana)
cols_simple=["Glucose", "BloodPressure", "BMI"] #cols con valores faltantes <5%

imputer_simple=SimpleImputer(strategy="median")
imputer_simple.fit(x_train_imputed[cols_simple]) #aprende el estadístico mediana
#de las columnas seleccionadas del conjunto x_train

x_train_imputed[cols_simple]=imputer_simple.transform(x_train_imputed[cols_simple])
#aplica el estadístico al conjunto x_train
x_test_imputed[cols_simple]=imputer_simple.transform(x_test_imputed[cols_simple])
#aplica al conjunto test la mediana de train

print("\n✅Imputación simple completada\n")

#----------------------IMPUTACIÓN KNN-----------------------
#Imputación KNN
cols_knn=["SkinThickness", "Insulin"]

imputer_knn=KNNImputer(n_neighbors=5)
imputer_knn.fit(x_train_imputed[cols_knn]) #aprende la estructura de las cols
#seleccionadas del conjunto x_train

x_train_imputed[cols_knn]=imputer_knn.transform(x_train_imputed[cols_knn])
#aplica el estadístico al conjunto x_train
x_test_imputed[cols_knn]=imputer_knn.transform(x_test_imputed[cols_knn])
#aplica la imputación basada en los vecinos más cercanos aprendidos en x_train

print("\n✅Imputación KNN completada\n")



#Comprobación de los resultado de no presencia de nulos
console.rule("[#00ffff]Datos finales[/#00ffff]",
             style="bright_magenta")
print(f"Valores nulos en x_train_imputed: {x_train_imputed.isnull().sum().sum()}")
print(f"Valores nulos en x_test_imputed: {x_test_imputed.isnull().sum().sum()}")

Tras la imputación no se observan valores faltantes en ninguna de las variables del dataset.

In [ ]:
#---------------------TABLA IMPUTACIÓN-----------------------
#Tabla resumen de la imputación
console.rule("[#00ffff]Tabla resumen de imputación[/#00ffff]",
             style="bright_magenta")

cols_null=["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

tabla_resumen=pd.DataFrame({ #le pasamos al nuevo df el diccionario con
                            #el contenido de cada columna
    "Variable":cols_null, #La lista de las variables problemáticas
    "% Valores faltantes": [
        round((x_train[col].isna().sum()/len(x_train))*100, 2)
        for col in cols_null #creación de una lista con los cálculos para %
        #valores faltantes para cada col en cols_null (variables problemáticas)
    ],
    "Antes imputación (Media ± SD)": [ #Media y desv std para cada una de las variables
        f"{x_train[col].mean():.2f} ± {x_train[col].std():.2f}"
        #en formato f-string para incluir texto y cálculos
        for col in cols_null
    ],
    "Después imputación (Media ± SD)": [
        f"{x_train_imputed[col].mean():.2f} ± {x_train_imputed[col].std():.2f}"
        for col in cols_null
    ],
    "Mediana antes": [
        round(x_train[col].median(), 2) #cálculo de la mediana con 2 decimales
        for col in cols_null
    ],
    "Mediana después": [
        round(x_train_imputed[col].median(), 2)#cálculo mediana con 2 decimales
        for col in cols_null
    ]
})
tabla_resumen


#---------------------TABLA A FIGURA-----------------------
#Transformación de la tabla en forma de figura para guardarla como imágen
fig, ax=plt.subplots(figsize=(12,4)) #creación figura horizontal
#ax.axis("tight") #adaptación del espacio a la tabla
ax.axis("off") #quitar los ejes (x, y)

#creación de la tabla
tabla=ax.table(
    cellText=tabla_resumen.values, #extracción de los valores de zero_table
    colLabels=tabla_resumen.columns, #col como encabezados
    cellLoc="center",
    colLoc="center",
    loc="center" #tabla en el centro del eje
)

tabla.auto_set_font_size(False) #Desactiva ajuste automático del tamaño letra
tabla.set_fontsize(10) #Se indica el tamaño letra deseado
tabla.scale(1, 1.5) #escalado del ancho y alto de la tabla, para
#ser más legible y no quede apretada

#Ajuste del estilo
for (fila, col), celda in tabla.get_celld().items(): #Bucle que pasa por cada una
#de las celdas, devolviendo un diccionario con posiciones (fila, col)
    celda.set_edgecolor('black') #bordes de la celda negro
    celda.set_linewidth(1) #grosor del borde

    #Comprobación de si la celda es el encabezado (fila==0)
    if fila == 0:
        celda.set_text_props(weight='bold', color='black') #Texto negrita+negro
        celda.set_facecolor('white') #fondo blanco
    else:
        celda.set_text_props(color='black') #si no es encabezado solo negro
        celda.set_facecolor('white') #fondo blanco

#Ajuste de los anchos de las cols manualmente
anchos = [0.15, 0.19, 0.31, 0.31, 0.2, 0.2] #lista de los anchos de cada col

for col, ancho in enumerate(anchos):
    for fila in range(len(tabla_resumen) + 1):  #+1 para contar el encabezado
    #al recorrer las filas
        tabla[(fila, col)].set_width(ancho) #asigna el ancho a cada fila según
        #la col que se encuentre

plt.savefig("tabla_resumen_imputación.png", bbox_inches="tight", dpi=300) #Guardado
#de la figura como imágen
plt.show()


La comparación de las estadísticas descriptivas antes y después muestra para las variables *Glucose*, *BloodPressure* y *BMI* cambios poco relevantes.

La imputación mediante la mediana preservó adecuadamente la distribución original.

La variable *SkinThickness* presenta cambios poco relevantes en la mediana y desviación estándard, a diferencia de *Insuline* que presenta cambios marcados en mediana y desviación estandard, y menos marcada en la media.

Los cambios son resultado de la mderada reducción de variabilidad, pero no altera de forma drástica la tendencia de las variables.

### 7. MANEJO DESBALANCEO DE CLASES
Como se ha observado en el countplot de balanceo de clases, el número de registros de outcome=0 es de 500 (65,10%), mientras que para Outcome=1 es de 268 (34,90%). Esto supone un desbalanceo de clases.

Para mejorar la sensibilidad del modelo, aplicaremos la técnica SMOTE (Synthetic Minority Over-sampling Technique) sobre el conjunto de entrenamiento para no contaminar el conjunto de test con puntos arificiales.  

El conjunto test debe mantener su distribución original. Se aplica después de la imputación ya que necesita valores completos para generar muestras sintéticas trabajando en el espacio de características original.

In [ ]:
#---------------------LIBRERÍAS-----------------------
#Carga de SMOTE desde librería
from imblearn.over_sampling import SMOTE


#---------------------SMOTE-----------------------
#Creación del objeto SMOTE dentro de la variable smote
smote=SMOTE(random_state=42) #smote con semilla fija, para reproducibilidad
#Aplicación de SMOTE a los datos de entrenamiento
#devolvera un nuevo x e y balanceados
x_train_smote, y_train_smote=smote.fit_resample(x_train_imputed, y_train)

console.rule("[#00ffff]Manejo desbalanceo de clases:[/#00ffff]",
             style="bright_magenta")


#---------------------VERIFICACIÓN-----------------------
#Verificación del balanceo
print("Distribución original en entrenamiento:")
print(y_train.value_counts())
print("\nDistribución después de SMOTE:")
print(pd.Series(y_train_smote).value_counts()) #cuenta cuantos ejemplos hay
#en cada clase

Al aplicar la técnica SMOTE se observa como las clases quedan equilibradas. Antes de SMOTE sla distribución original es de *Outcome*=0 400 registros y, *Outcome*=1 214 registros. Después de SMOTE la distribución de *Outcome*=0 es de 400 registros y *Outcome*=1 de 400 registros.

Mejora la capacidad para detectar la clase minoritaria de casos positivos.

### 8. NORMALIZACIÓN / ESCALADO
Se aplica el escalado mediante StandardScaler para normalizar las variables.

La importancia del escalado reside en los modelos sensibles a la escala de los datos, como Support Vector Classifier (SVC), que son sensibles a escalas.

Se aplica el escalado a los demás algoritmos para seguir un pipeline homogéneo.

In [ ]:
#---------------------LIBRERÍAS-----------------------
#Escalado de datos mediante StandardScaler
from sklearn.preprocessing import StandardScaler


#---------------------ESCALADO-----------------------
scaler=StandardScaler()
scaler.fit(x_train_smote)

#Ajuste del scaler solo con datos balanceados de entrenamiento
x_train_scaled=scaler.transform(x_train_smote)

#Transformación de los datos de test con los parámetros aprendidos en train
x_test_scaled=scaler.transform(x_test_imputed)

#Convertir a DataFrame para mantener los nombres de las columnas
x_train_scaled=pd.DataFrame(x_train_scaled, columns=x_train_imputed.columns,
                            index=x_train_smote.index)
x_test_scaled=pd.DataFrame(x_test_scaled, columns=x_test_imputed.columns,
                           index=x_test_imputed.index)


#---------------------VISUALIZACIÓN-----------------------
console.rule("[#00ffff]Comprobación del Escalado[/#00ffff]",
             style="bright_magenta")
print(f"Media de variables escaladas (~0):")
print(x_train_scaled.mean().round(3))
print(f"\nDesviación estándar de variables escaladas (~1):")
print(x_train_scaled.std().round(3))

El resultado del escalado muestra que la media de las variables ahora son cercanas a 0, y que sus desviaciones estándard son cercanas a 1.

### 9. ENTRENAMIENTO DE MODELOS
El balanceo de clases se ha realizado previamente con SMOTE sobre el conjunto de entrenamiento, por lo que todos los modelos se entrenan sobre datos balanceados.

Se prueban 4 algoritmos: Regresión Logística, Random Forest, SVM, y XGB Classifier. Se escogen estos algoritmos por lo siguiente:
- Regresión Logística es un modelo lineal interpretable y es el baseline.
- Random Forest presenta un ensamble robusto y es capaz de captar no linealidad.
- SVC es fuerte en problemas complejos.
- XGBoost tiene un boosting potente y es habitual en clasificación tabular.


In [ ]:
#---------------------LIBRERÍAS-----------------------
#Carga de las librerías necesarias para los modelos y sus métricas de rendimiento
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
import time
import warnings
warnings.filterwarnings("ignore")


#---------------------MODELOS-----------------------
#Definición de los modelos guardandolos en el diccionario "models"
models={
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42,
                                               C=1.0), #regularización por defecto
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10,
                                            min_samples_split=5,
                                            min_samples_leaf=2,
                                            random_state=42),
    "SVC": SVC(kernel="rbf", probability=True, C=1.0, gamma="scale",
               random_state=42), #no lineal
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                             random_state=42,
                             use_label_encoder=False,
                             eval_metric="logloss",
                             verbosity=0)
}

#Creación de una lista para almacenar los resultados
results_list=[]

#Creación de diccionario para guardar los modelos entrenados
model_trained={}

console.rule("[#00ffff]Entrenamiento de modelos:[/#00ffff]",
             style="bright_magenta")


#---------------------ENTRENAMIENTO Y EVALUACIÓN-----------------------
for name, model in models.items():
    console.print(Panel(f"[cyan]Entrenando {name}[/cyan]", border_style="magenta",
                        padding=(1))) #mostrar progreso con panel
    start_time=time.time() #medición del tiempo de entrenamiento
    model.fit(x_train_scaled, y_train_smote) #entramiento del modelo
    training_time=time.time()-start_time #calculo del tiempo de entrenamiento
    model_trained[name]=model #guardar el modelo entrenado

    #Predicciones
    y_pred=model.predict(x_test_scaled) #realización de predicciones
    y_pred_proba=model.predict_proba(x_test_scaled)[:, 1] #probabilidad para la
    #clase 1

    #Cálculo de las métricas
    accuracy=accuracy_score(y_test, y_pred)
    precision=precision_score(y_test, y_pred)
    recall=recall_score(y_test, y_pred)
    f1=f1_score(y_test, y_pred)
    roc_auc=roc_auc_score(y_test, y_pred_proba)

    #Matriz de confusión
    conf_matrix=confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp=conf_matrix.ravel()

    #Cálculo de la especificidad
    specificity=tn/(tn+fp) if (tn+fp)>0 else 0

    #Guardar resultados en el diccionario creado anteriormente
    results_list.append({"Modelo":name, "Accuracy": round(accuracy, 4),
                         "Precision": round(precision, 4),
                         "Recall (Sensibilidad)": round(recall, 4),
                         "F1-score": round(f1, 4),
                         "ROC-AUC": round(roc_auc, 4),
                         "Confusion Matrix": conf_matrix,
                         "Specificity": round(specificity, 4),
                         "Tiempo (s)": round(training_time, 4),
                         "True Negative": tn,
                         "True Positive": tp,
                         "False Negative": fn,
                         "False Positive": fp})

In [ ]:
#---------------------DATAFRAME DE RESULTADOS-----------------------
#Se convierte la lista creada anteriormente a df
results_df=pd.DataFrame(results_list)

#Se ordenan los resultados según la métrica principal Recall
results_df=results_df.sort_values("Recall (Sensibilidad)",
                                    ascending=False).reset_index(drop=True)

#Especificación de formato para mejorar visualización
styled_results_df=(results_df.style.format({
    "Accuracy": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall (Sensibilidad)": "{:.4f}",
    "F1-score": "{:.4f}",
    "ROC-AUC": "{:.4f}",
    "Specificity": "{:.4f}",
    "Tiempo (s)": "{:.1f}"
}).highlight_max(
    subset=["Recall (Sensibilidad)", "F1-score", "ROC-AUC"],
    color="green")
  .set_properties(**{"text-align":"center"})
  )

display(styled_results_df)

## 10. EVALUACIÓN INICIAL DE LOS MODELOS

Observación visual inicial de los modelos a partir de tablas y gráficos para su evaluación y comparación. Se realiza antes de la validación y optimización.

In [ ]:
#---------------------RESUMEN ESTADÍSTICO-----------------------
#Resumen de estadísticas descriptivas iniciales de algunas métricas antes de
#la exploración visual
console.rule("[#00ffff]Resumen Estadístico de los Resultados[/#00ffff]",
             style="bright_magenta")

print("\nEstadísticas descriptivas métricas:\n")
print(
    f"Recall (Sensibilidad): media={results_df["Recall (Sensibilidad)"].mean():.4f} "
    f"(min={results_df["Recall (Sensibilidad)"].min():.4f}, "
    f"max={results_df["Recall (Sensibilidad)"].max():.4f})\n")
print(
    f"ROC-AUC: media={results_df["ROC-AUC"].mean():.4f} "
    f"(min={results_df["ROC-AUC"].min():.4f}, "
    f"max={results_df["ROC-AUC"].max():.4f})\n")
print(
    f"Tiempo medio entrenamiento medio:{round(results_df["Tiempo (s)"].mean(), 1)}s")

In [ ]:
#---------------------TABLA COMPARATIVA RESULTADOS-----------------------
console.rule("[#00ffff]Resultados Comparativos de los Modelos:[/#00ffff]",
             style="bright_magenta")
print("\n")

#Definición de las métricas que más nos interesan para ponerlas en una tabla
#para la memoria del TFM
tabla_tfm=results_df[["Modelo", "Recall (Sensibilidad)", "ROC-AUC", "F1-score",
                      "Accuracy", "Precision", "Specificity"]].copy()

tabla_tfm=tabla_tfm.round(4)

#Orden por valores de sensibilidad
tabla_tfm=tabla_tfm.sort_values("Recall (Sensibilidad)",
                                ascending=False).reset_index(drop=True)

#Definición de una función para establecer el umbral de colores de las métricas
#de forma relativa
#Verde=Mejor valor, Amarillo=Segundo mejor valor, Rojo=Resto de valores
def color_metrica_relativa(valor, metrica, df):
    valores_ordenados=sorted(df[metrica].unique(), reverse=True)
    if valor == valores_ordenados[0]:
        return "#C6EFCE" #verde claro
    elif len(valores_ordenados) > 1 and valor == valores_ordenados[1]:
        return "#FFFACD" #amarillo claro
    else:
        return "#FFC0CB" #rojo claro


#Creación de la tabla
fig, ax = plt.subplots(figsize=(13, 3.8))
ax.axis("off")

table = ax.table(
    cellText=tabla_tfm.values,
    colLabels=tabla_tfm.columns,
    loc="center",
    cellLoc="center")

table.auto_set_font_size(False) #Desactiva ajuste automático del tamaño letra
table.set_fontsize(11) #Se indica el tamaño letra deseado
table.scale(1, 2.5) #escalado del ancho y alto de la tabla, para

for (fila, col), celda in table.get_celld().items():
    celda.set_edgecolor("black") #bordes de la celda negro
    celda.set_linewidth(1) #grosor del borde
    celda.set_height(0.20) #altura filas
    #Definición del estilo de la cabecera
    if fila == 0:
        celda.set_text_props(weight="bold", color="black") #Texto negrita+negro
        celda.set_facecolor("#D9EAF7") #color azul claro para los encabezados
    else:
        nombre_col=tabla_tfm.columns[col]

        if nombre_col != "Modelo":
            valor=tabla_tfm.iloc[fila-1, col]
            celda.set_facecolor(color_metrica_relativa(valor, nombre_col, tabla_tfm))
        else:
          celda.set_facecolor("#D9EAF7") #azul claro para la columna de modelos
          celda.set_text_props(weight="bold", color="black")

#Ajuste de los anchos de las cols manualmente
anchos = [0.2, 0.25, 0.14, 0.14, 0.14, 0.14, 0.14] #lista de los anchos de cada col

#Recirre las celdas existentes para ccnfigurar los anchos
for (fila, col_idx), celda in table.get_celld().items():
    if col_idx < len(anchos): #así se asegura que el index de la col está dentro
    #de los límites de los anchos
        celda.set_width(anchos[col_idx]) #asigna el ancho a cada fila según
        #la col que se encuentre

plt.title("Comparativa de modelos de clasificación", fontsize=14,
          fontweight="bold", pad=20)
plt.savefig("comparativa_modelos.png", dpi=300, bbox_inches="tight")
plt.show()

La coloración de la tabla es relativa entre modelos, siendo el verde el mejor valor de cada métrica, el amarillo el segundo valor, y el rojo el resto de valores.

Se usa este código de colores a modo de ayuda visual para facilitar la comparación.

Según la métrica de sensibilidad se observa que el mejor modelo de predicción es Random Forest y SVC.

Los modelos pueden detectar correctamente el 72,2% de los casos reales de diabetes.

El modelo que presenta más métricas con mayores resultados es SVC. Presenta mayor F1-score, accuracy, precision y specificity.

In [ ]:
#---------------------MATRIZ DE CONFUSIÓN-----------------------
console.rule("[#00ffff]Matriz de confusión:[/#00ffff]",
             style="bright_magenta")
print("\n")

#Creación de df para convertirlo posteriormente a una tabla
tabla_mc=results_df[["Modelo", "False Negative", "False Positive", "True Negative",
                  "True Positive"]].copy()

#Crear una figura
fig, ax=plt.subplots(figsize=(12, 3.8))
ax.axis("off")

#Creación de la tabla
table=ax.table(cellText=tabla_mc.values, colLabels=tabla_mc.columns,
                  loc="center", cellLoc="center")

#Definicón del estilo de la tabla
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

#Se pone el valor mínimo de False Negative para así destacarlo
min_fn=tabla_mc["False Negative"].min()

#Aplicación de los estilos de la tabla
for (fila, col), celda in table.get_celld().items():
    celda.set_edgecolor("black")
    celda.set_linewidth(1)
    celda.set_height(0.20)

    #Definición del estilo de la cabecera en color azul claro y en negrita
    if fila==0:
        celda.set_text_props(weight="bold", color="black")
        celda.set_facecolor("#D9EAF7")
    else: #si no es la cabecera, observa el nombre de la col para seguir las
    #siguientes instrucciones del estilo
        nombre_col=tabla_mc.columns[col]
        valor=tabla_mc.iloc[fila - 1, col]

        #Definicón del estilo de la col Modelo, azul claro y negrita
        if nombre_col=="Modelo":
            celda.set_facecolor("#D9EAF7")
            celda.set_text_props(weight="bold", color="black")
        #Definicón del estilo de la col False Negative destacada
        elif nombre_col=="False Negative":
            celda.set_text_props(weight="bold", color="black") #negrita y negra
            if valor==min_fn:
                celda.set_facecolor("#C6EFCE")   #si es igual al mínimo del
                #valor de False Negative, se indicara en verde claro
            else:
                celda.set_facecolor("#FFC0CB")   #si es diferente a ese valor
                #la celda será de color rojo claro
        #El resto de columnas
        else:
            celda.set_facecolor("#F5F5F5")
            celda.set_text_props(weight="bold", color="black")

#Ajuste manual de anchos
anchos = [0.20, 0.22, 0.19, 0.19, 0.19]

for (fila, col_idx), celda in table.get_celld().items():
    if col_idx < len(anchos):
        celda.set_width(anchos[col_idx])

plt.title("Matriz de confusión de cada uno de los modelos",
          fontsize=14, fontweight="bold", pad=20)

plt.savefig("matriz_confusion_modelos.png", dpi=300, bbox_inches="tight")
plt.show()

Random Forest y SVC son los modelos con menor número de falsos negativos (15).

In [ ]:
#----------------------MATRIZ CONFUSIÓN HEATMAP--------------
#Carga de librerias para la matriz de confusión
from sklearn.metrics import ConfusionMatrixDisplay

console.rule("[#00ffff]Matrices de confusión visuales:[/#00ffff]",
             style="bright_magenta")

#Creación de la figura
fig, axes=plt.subplots(2, 2, figsize=(10, 10))
fig.suptitle("Matrices de confusión por modelo", fontsize=16, fontweight="bold")

#Bucle para especificar los tn, fp, fn y tp de cada modelo
for ax, (_, fila) in zip(axes.flatten(), results_df.iterrows()):
    cm=np.array([[fila["True Negative"], fila["False Positive"]],
                   [fila["False Negative"], fila["True Positive"]]])
    disp=ConfusionMatrixDisplay(confusion_matrix=cm,
                                display_labels=["No Diabetes", "Diabetes"])
    disp.plot(ax=ax, colorbar=False, cmap="Oranges",
              text_kw={"fontsize": 20, "fontweight": "bold"})
    ax.set_title(fila["Modelo"], fontweight="bold", fontsize=15)

plt.tight_layout()
plt.savefig("matrices_confusion_visual.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#---------------------MEJOR MODELO SEGÚN RECALL-----------------------
#Obtención del valor máximo de la métrica recall
max_recall=results_df["Recall (Sensibilidad)"].max()

#Filtración de las filas de result_df que el Recall sea igual al máximo recall
#así vemos todos los modelos que tienen el mejor recall
modelos_mejor_recall=results_df[
    results_df["Recall (Sensibilidad)"]==max_recall].copy()

console.rule("[#00ffff]Análisis de los modelos con mayor Recall:[/#00ffff]",
             style="bright_magenta")

print("\n")

#Resultado del valor máximo de recoll y su %
print(f"Máximo Recall (Sensibilidad):{max_recall:.4f}({max_recall*100:.1f})\n")

#Mostración de los modelos con mejor recall
print("Modelos con mejor Sensibilidad:")
#Se lee cada fila como si fuera una serie, representado el índice de la fila por "_"
for _, fila in modelos_mejor_recall.iterrows():
  print(f"- {fila["Modelo"]}")
  print(f"  Falsos Negativos: {fila["False Negative"]}")
  print(f"  ROC-AUC: {fila["ROC-AUC"]:.4f}")
  print(f"  Especificidad: {fila["Specificity"]:.4f}")
  print(f"  F1-score:{fila["F1-score"]:.4f}\n")

#Las condiciones del recall para que se clasfique en una categoría
if max_recall > 0.85:
    nivel_sensibilidad="Excelente"
elif max_recall > 0.75:
    nivel_sensibilidad="Aceptable"
else:
    nivel_sensibilidad= "Mejorable"

print(f"El nivel de sensibilidad de los mejores modelos es {nivel_sensibilidad}.")

Según la métrica de sensibilidad se observa que el mejor modelo de predicción es Random Forest y SVC.

Aún así, el nivel de sensibilidad podría ser mejorable, ya que no llega al 0,85.

In [ ]:
#---------------------GRÁFICO COMPARATIVO 4 MODELOS-----------------------
console.rule("[#00ffff]Visualizaciones Comparativas de los 4 Modelos:[/#00ffff]",
             style="bright_magenta")

#Configuración del estilo general de los gráficos
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("Set2")

#Creación de una figura con 4 subgráficos distribuidos en 2 filas y 2 columnas
fig, axes=plt.subplots(2, 2, figsize=(14, 10))

#Título general de la figura
fig.suptitle("Análisis Comparativo de Modelos de Clasificación",
             fontsize=20,
             fontweight="bold")


#---------------------GRÁFICO 1: RECALL VS ROC-AUC-----------------------
ax1=axes[0, 0] #selección del primer eje de la figura
x=np.arange(len(results_df)) #array que va determinar la posición de cada modelo
#en el eje x
width=0.25 #ancho de cada barra

#Primer gráfico de los valores de Recall vs ROC-AUC para cada uno de los modelos
#Creación de las barras para Recall
bars1=ax1.bar(x-width/2,
              results_df["Recall (Sensibilidad)"],
              width,
              label="Recall (Sensibilidad)",
              color="#E69F00",
              edgecolor="black")

#Creación de las barras para ROC-AUC
bars2=ax1.bar(x+width/2,
              results_df["ROC-AUC"],
              width,
              label="ROC-AUC",
              color="#56B4E9",
              edgecolor="black")

#Etiquetas, título y leyenda del gráfico
ax1.set_xlabel("Modelos") #Nombre del eje de las x como modelos
ax1.set_ylabel("Puntuación") #nombre del eje y que será la puntuación de las dos
#métricas
ax1.set_title("Comparación entre Recall y ROC-AUC") #Título del gráfico
ax1.set_xticks(x)
ax1.set_xticklabels(results_df["Modelo"], rotation=45, ha="right")
ax1.legend(loc="lower right")
ax1.set_ylim(0, 1.05)
ax1.grid(True, alpha=0.3)

#Especificación de los valores numéricos de cada barra de Recall
for bar in bars1:
    height=bar.get_height()
    ax1.annotate(f"{height:.3f}", xy=(bar.get_x()+bar.get_width()/2, height),
                 xytext=(0, 3), textcoords="offset points", ha="center",
                 va="bottom", fontsize=10)

#Valores numéricos para las barras de ROC-AUC
for bar in bars2:
  height=bar.get_height()
  ax1.annotate(f"{height:.3f}",
               xy=(bar.get_x()+bar.get_width()/2, height), xytext=(0, 3),
               textcoords="offset points", ha="center", va="bottom",
               fontsize=10)


#---------------------GRÁFICO 2: FALSOS NEGATIVOS-----------------------
#Gráfico 2 comparando el valor de los Falsos Negativos de los modelos
ax2=axes[0, 1] #segundo eje de la figura

#El modelo con menos falsos negativos se configura en rosa
colors_fn=["#CC79A7" if valor==results_df["False Negative"].min() else "#95a5a6"
           for valor in results_df["False Negative"]]

#Para facilitar la comparación se representan en un gráfico horizontal
bars=ax2.barh(results_df["Modelo"],
              results_df["False Negative"],
              color=colors_fn,
              edgecolor="black")

#Título y etiquetas
ax2.set_xlabel("Número de Falsos Negativos")
ax2.set_title("Falsos Negativos")
ax2.grid(True, alpha=0.3, axis="x")

#Especificación de los valores numéricos de cada barra
for bar, val in zip(bars, results_df["False Negative"]):
    ax2.annotate(f"{val}", xy=(bar.get_width(), bar.get_y()+bar.get_height()/2),
                 xytext=(5, 0), textcoords="offset points", ha="left",
                 va="center")


#---------------------GRÁFICO 3: HEATMAP MÉTRICAS-----------------------
#Gráfico 3 de un heatmap de las metricas
#Selección del tercer eje de la figura
ax3=axes[1, 0]
#Dataframe para especificar las métricas numéricas para comparar
metricas_plot=results_df[["Modelo", "Recall (Sensibilidad)", "ROC-AUC",
                            "F1-score", "Accuracy", "Precision",
                            "Specificity"]].copy()

metricas_plot=metricas_plot.set_index("Modelo") #la col modelo será el índice
#del heatmap

sns.heatmap(metricas_plot, annot=True, cmap="YlGnBu", fmt=".4f", ax=ax3,
            cbar_kws={"label": "Puntuación"})
ax3.set_title("Heatmap de las métricas de los modelos")


#---------------------GRÁFICO 4: F1-SCORE-----------------------
#Gráfico 4 de comparación de los F1-score de los modelos
ax4=axes[1, 1] #cuarto eje de la figura

#El mayor f1-score se indica en verde
colors_f1=["#2ecc71" if valor==results_df["F1-score"].max() else "#3498db"
             for valor in results_df["F1-score"]]

#Gráfico con barras horizontales
bars=ax4.barh(results_df["Modelo"],
              results_df["F1-score"],
              color=colors_f1,
              edgecolor="black")

ax4.set_xlabel("F1-score") #leyenda del eje de las x
ax4.set_title("Comparación del F1-Score por Modelo") #título del gráfico
ax4.set_xlim(0, 1)
ax4.grid(True, alpha=0.3, axis="x")

#Valor numérico de cada barra (f1-score)
for bar, val in zip(bars, results_df["F1-score"]):
    ax4.annotate(f"{val:.2f}s",
                 xy=(bar.get_width(), bar.get_y()+bar.get_height()/2),
                 xytext=(5, 0),
                 textcoords="offset points",
                 ha="left",
                 va="center")

#Ajuste automático de los espacios de la figura
plt.tight_layout()

#Guardar la figura en un archivo .png
plt.savefig("comparativa_modelos_global.png",
            dpi=300,
            bbox_inches="tight",
            facecolor="white")

#Mostrar la figura final
plt.show()

Random Forest obtiene el mejor valor de ROC-AUC y se utiliza para representar la curva ROC y analizar la importancia de las variables.

Se observa también que Regresión Logística es el modelo con el peor rendimiento.

In [ ]:
#---------------------CURVA ROC DEL MEJOR MODELO-----------------------
#Observación de la curva ROC del mejor modelo
#Carga de librerías necesarias
from sklearn.metrics import roc_curve, auc

modelo_mejor_auc=results_df.loc[results_df["ROC-AUC"].idxmax(), "Modelo"]
modelo_auc=model_trained[modelo_mejor_auc]

#Cálculo de las probabilidades que se han predicho
y_pred_proba_auc=modelo_auc.predict_proba(x_test_scaled)[:, 1]

#Cálculo de los puntos de la curva ROC
fpr, tpr, thresholds=roc_curve(y_test, y_pred_proba_auc)

#Obtención del AUC del mejor modelo a partir del df de resultados
roc_auc_best=results_df.loc[results_df["ROC-AUC"].idxmax(), "ROC-AUC"]

#Creación de la figura
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#2ecc71', lw=2,
         label=f"Curva ROC (AUC={roc_auc_best:.4f})")

#Representación de la línea diagonal de un clasificador aleatorio
plt.plot([0, 1], [0, 1], color='#95a5a6', lw=2, linestyle='--',
         label="Clasificador Aleatorio")

#Ejes, etiquetas y título
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("Tasa de Falsos Positivos (1 - Especificidad)", fontsize=12)
plt.ylabel("Tasa de Verdaderos Positivos (Sensibilidad)", fontsize=12)
plt.title(f"Curva ROC-{modelo_mejor_auc}", fontsize=14, fontweight="bold")
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)

plt.savefig("curva_roc_mejor_auc.png", dpi=300, bbox_inches="tight",
            facecolor='white')
plt.show()

Random Forest presenta una curva ROC de 0,8293, que indica una buena capacidad para discriminar entre pacientes positivos y negativos para diabetes.

La curva Roc está situada por encima de la línea del clasificador aleatorio, así que el modelo tiene capacidad predictiva.

Se puede extrar que el modelo puede mantener buen equilibrio entre sensibilidad y tasa de falsos positivos.

Como conslusión, Random Forest es un modelo adecuado.

In [ ]:
#---------------------IMPORTANCIA DE VAIRABLES EN MODELOS BASADOS---------------
#--------------------------------------EN ÁRBOLES---------------------------
#Detección de las características más importante para modelos basados en árboles
#como Random Forest
console.rule("[#00ffff]Importancia de las Características (Variables Predictoras):[/#00ffff]",
             style="bright_magenta")

modelo_interpretable="Random Forest"
modelo_arbol=model_trained[modelo_interpretable]

#función para determinar si un objeto presenta una propiedad o no.
#Hace el análisis si el modelo permite medir la importancia de las variables
if hasattr(modelo_arbol, "feature_importances_"):
    feature_importance=pd.DataFrame({
        "Característica": x_train_scaled.columns,
        "Importancia": modelo_arbol.feature_importances_
    }).sort_values("Importancia", ascending=True) #df que almacena todas las
    #variables con el peso dado por el modelo, ordenadas de mayor a menor
    #importancia

#Creación de figura para visualización de las características
plt.figure(figsize=(7, 4))
colors=plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(feature_importance)))

plt.barh(feature_importance["Característica"], feature_importance["Importancia"],
             color=colors, edgecolor="black")
plt.xlabel("Importancia", fontsize=12)
plt.ylabel("Característica", fontsize=12)
plt.title(f"Importancia de Características-{modelo_interpretable}",
          fontsize=14, fontweight="bold")
plt.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("importancia_variables_RF.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()


#Creación de tabla resumen
imp_table=Table(title=f"Variables predictoras más relevantes-{modelo_interpretable}",
                  box=box.ROUNDED)
imp_table.add_column("Característica", style="cyan")
imp_table.add_column("Importancia", justify="center", style="green")
imp_table.add_column("Importancia Relativa (%)", justify="center")

max_imp = feature_importance["Importancia"].max()

for _, row in feature_importance.sort_values("Importancia", ascending=False).iterrows():
    imp_table.add_row(row["Característica"], f"{row["Importancia"]:.4f}",
                    f"{row["Importancia"]/max_imp*100:.1f}%")
console.print(imp_table)

La variable *"Glucose"* es el predictor más relevante. Este resultado es consistente con los criterios diagnósticos para la diabetes, ya que los niveles de glucosa es el principal criterio.

En segundo lugar, está *"BMI"*, que coincide con uno de los factores de riesgo para la diabete, reflejando la relación entre condiciones de obesidad y diabetes tipo II.

Por último, en tercer lugar *"Age"*, presenta una importancia moderada ya que es un factor de riesgo ampliamente conocido.

Cabría esperar que *"Insulin"* se encontrará en el ranking de importancia. La posición de relevancia que ocupa *"Insulin"* podría deberse a la menor calidad de la variable. Como se ha visto en el paso de valores faltantes, esta variable presentaba un 48,7% de *missing values*.

Se puede afirmar que el modelo basa sus deciones en variables clínicamente relevantes, aportando coherencia e interpretabilidad.

## 11. VALIDACIÓN

In [ ]:
#------------------------VALIDACIÓN------------------------------
#Ejecución de la validación cruzada mediante Stratified-K Fold
#Carga de librerías
from sklearn.model_selection import StratifiedKFold, cross_val_score

console.rule("[#00ffff]Validación Cruzada (5-Fold)[/#00ffff]",
             style="bright_magenta")

#Definición de la estrategia de validación
validation=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

#Evaluación de los modelos mediante validación cruzada
name_models=[] #creación de una lista donde se almacenaran los resultados
v_means=[]
v_stds=[]

for name, model in models.items():
    #Cálculo de la métrica Recall en validación cruzada
    scores=cross_val_score(model, x_train_scaled, y_train_smote,
                                    cv=validation, scoring="recall", n_jobs=-1)

    v_means.append(scores.mean())
    v_stds.append(scores.std())
    name_models.append(name)
    print(f"{name:<25} | Recall V:{scores.mean():.4f} (±{scores.std():.4f})")

#Ordenación de los modelos y estadísticos según rendimiento, para incorporar en
#gráfico
sorted_id=np.argsort(v_means)[::-1]
name_model_sorted=[name_models[i] for i in sorted_id]
v_means_sorted=[v_means[i] for i in sorted_id]
v_stds_sorted=[v_stds[i] for i in sorted_id]

#Gráfico de barras horizontales
fig, ax=plt.subplots(figsize=(10, 5))
colors=plt.cm.RdYlGn(np.linspace(0.3, 0.8, len(name_model_sorted)))[::-1]# Colores según rendimiento (verde para mejor, rojo para peor)
bars=ax.barh(name_model_sorted, v_means_sorted, xerr=v_stds_sorted,
             color=colors, edgecolor="black", linewidth=1, capsize=5,
             error_kw={"linewidth": 1.5, "ecolor": "black"})

#Especificación de los valores de la media y desv std al final de las barras
for bar, mean, std in zip(bars, v_means_sorted, v_stds_sorted):
    ax.text(mean + std + 0.01, bar.get_y() + bar.get_height()/2,
            f"{mean:.3f} ± {std:.3f}",
            va="center", fontsize=9, fontweight="bold")

#Determinación de umbral de buen rendimiento
ax.axvline(x=0.80, color="green", linestyle="--", linewidth=1.5, alpha=0.7)

#Configuración gráfico
ax.set_xlabel("Recall (Sensibilidad)", fontsize=11, fontweight="bold")
ax.set_title("Validación Cruzada (5-fold): Recall por Modelo (con desv std)")
ax.set_xlim(0.55, 1.0)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("validacion_cruzada.png", dpi=300, bbox_inches="tight")
plt.show()

console.print(
    Panel(
        f"Modelo con mayor recall medio en validación cruzada: [cyan]{name_model_sorted[0]}[/cyan]\n"
        f"Recall medio: {v_means_sorted[0]:.4f} (±{v_stds_sorted[0]:.4f})",
        border_style="magenta",
        padding=(1)
    )
)

Random Forest presenta el mayor recall medio (0,85), es decir, mejor capacidad para detectar casos positivos de diabetes.

SVC y XGBoost casi están a la par, pero SVC un poco inferior.

La que presenta peor rendimiento es Regresión Logística. Es la que el rango de variabilidad es más amplio, y por lo tanto menor estabilidad.

Por esta razón, se elige Random Forest como candidato para la optimización.

### 12. OPTIMIZACIÓN DEL MEJOR MODELO

In [ ]:
#-----------------------OPTIMIZACIÓN DEL MEJOR MODELO-----------------
#Optimización del mejor modelo mediante GridSearchCV
from sklearn.model_selection import GridSearchCV

console.rule("[#00ffff]Optimización del mejor modelo[/#00ffff]",
             style="bright_magenta")

# Identificar el mejor modelo según Recall (de tus resultados)
mejor_nombre=name_model_sorted[0]
mejor_modelo_2= models[mejor_nombre]

print(f"\nOptimizando:{mejor_nombre}")

#Definición de los parámetros según el modelo
if "Random Forest" in mejor_nombre:
    param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [10, 15, None],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2]
    }
elif "XGBoost" in mejor_nombre:
    param_grid = {
        "n_estimators": [100, 200],
        "max_depth": [3, 6],
        "learning_rate": [0.01, 0.1]
    }
elif "SVC" in mejor_nombre:
    param_grid = {
        "C": [0.1, 1, 10],
        "gamma": ["scale", "auto"],
        "kernel": ["rbf"]
    }
elif "Logistic Regression" in mejor_nombre:
    param_grid = {
        "C": [0.1, 1, 10],
        "solver": ["liblinear", "lbfgs"]
    }
else:
    param_grid = {}  #si no se identifica, no optimizar

#GridSearch
grid=GridSearchCV(mejor_modelo_2, param_grid, cv=5, scoring="recall", n_jobs=-1)
grid.fit(x_train_scaled, y_train_smote)

console.rule("[#00ffff]Mejores parámetros[/#00ffff]", style="bright_magenta")
for param, valor in grid.best_params_.items():
    print(f"{param}: {valor}")

print(f"Mejor Recall (CV): {grid.best_score_:.4f}")

#Mejor modelo optimizado
modelo_optimizado=grid.best_estimator_

#Evaluación modelo optimizado en test
modelo_original=model_trained[mejor_nombre]
y_pred_original=modelo_original.predict(x_test_scaled)
recall_original_test=recall_score(y_test, y_pred_original)

#Evaluación del modelo optimizado en test
y_pred_opt=modelo_optimizado.predict(x_test_scaled)
recall_opt_test=recall_score(y_test, y_pred_opt)

console.rule("[#00ffff]Comparación:[/#00ffff]", style="bright_magenta")
print(f"Recall original (CV): {recall_original_test:.4f}")
print(f"Recall optimizado (test): {recall_opt_test:.4f}")

if recall_opt_test > recall_original_test:
    mejora=recall_opt_test - recall_original_test
    print(f"Mejora en test: +{mejora*100:.2f}% ✅")
    modelo_final=modelo_optimizado
    print("\nSe utilizará el modelo OPTIMIZADO")
else:
    print("No se observa mejora en test")
    modelo_final=modelo_original
    print("\nSe utilizará el modelo ORIGINAL")

print(f"\nModelo final seleccionado: {mejor_nombre}")
print(f"Recall final en test: {max(recall_original_test, recall_opt_test):.4f}")
print("\n✅ Optimización completada")

La optimización mediante GridSearch mejoró un poco el recall de la validación cruzada (0,86). Esto se debe a un mejor ajuste interno.

Sin embargo, al evaluar el modelo optimizado sobre el conjunto de prueba, no se observan mejoras respecto al modelo original.

Puede ser debido a un sobreajuste durante la optimización.

Dado estos datos, se mantiene el modelo original, ya que presenta un rendimiento más consistente en test.

###13. IMPLEMENTACIÓN Y DESPLIEGUE

In [ ]:
#------------------------IMPLEMENTACIÓN Y DESPLIEGUE-------------------
#Implementación y despliegue del modelo
#Carga de librerías
import joblib

console.rule("[#00ffff]Guargar el Modelo para el despliegue[/#00ffff]",
             style="bright_magenta")

#Guardado del modelo final, el que ha sido seleccionado tras la optimización
joblib.dump(modelo_final, "mejor_modelo_diabetes.pkl")
print("✅ Modelo guardado como mejor_modelo_diabetes.pkl")

#Guardado del scaler
joblib.dump(scaler, "scaler_diabetes.pkl")
print("✅ Scaler guardado como scaler_diabetes.pkl")

#Guardado imputadores
joblib.dump(imputer_simple, "imputer_simple_diabetes.pkl")
joblib.dump(imputer_knn, "imputer_knn_diabetes.pkl")
print("✅ Imputadores guardados correctamente")

#Función de predicción para implementación con nuevos pacientes
def predecir_diabetes(paciente):
    """
    Función para predecir diabetes en un nuevo paciente.
    paciente: diccionario con las 8 variables clínicas
    """
    #Carga de los componentes definidos anteriormente
    mejor_modelo_final=joblib.load("mejor_modelo_diabetes.pkl")
    scaler_cargado=joblib.load("scaler_diabetes.pkl")
    imputer_simple_cargado=joblib.load("imputer_simple_diabetes.pkl")
    imputer_knn_cargado=joblib.load("imputer_knn_diabetes.pkl")

    paciente_df=pd.DataFrame([paciente]) #conversión a df
    columnas=["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
              "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"] #orden de
              #las cols
    paciente_df=paciente_df[columnas]

    #Conversión de los ceros no plausibles a NaNs
    cols_cero_no_plausibles=["Glucose", "BloodPressure", "SkinThickness",
                             "Insulin", "BMI"]
    paciente_df[cols_cero_no_plausibles]=paciente_df[cols_cero_no_plausibles].replace(
        0, np.nan
    )

    #Imputación de los valores faltantes/ceros no pausibles clínicamente
    cols_simple=["Glucose", "BloodPressure", "BMI"]
    cols_knn=["SkinThickness", "Insulin"]

    #Imputación simple
    paciente_df[cols_simple]=imputer_simple_cargado.transform(paciente_df[cols_simple])
    #Imputación knn
    paciente_df[cols_knn]=imputer_knn_cargado.transform(paciente_df[cols_knn])

    #Escalado
    paciente_scaled=scaler_cargado.transform(paciente_df)

    #Predecicción
    pred=mejor_modelo_final.predict(paciente_scaled)[0]
    proba=mejor_modelo_final.predict_proba(paciente_scaled)[0][1]

    return {
        "Diagnostico": "Diabetes" if pred == 1 else "No Diabetes",
        "Probabilidad": f"{proba*100:.1f}%",
        "Riesgo": "Alto" if proba > 0.7 else "Medio" if proba > 0.3 else "Bajo"
    }

#Ejemplo de cómo se usaría
console.rule("[#00ffff]Ejemplo de predicción:[/#00ffff]", style="bright_magenta")

paciente_ejemplo={"Pregnancies": 3, "Glucose": 135, "BloodPressure": 75,
                  "SkinThickness": 30, "Insulin": 0, "BMI": 31.5,
                  "DiabetesPedigreeFunction": 0.85, "Age": 45}

resultado=predecir_diabetes(paciente_ejemplo)

print(f"Diagnóstico: {resultado["Diagnostico"]}")
print(f"Probabilidad: {resultado["Probabilidad"]}")
print(f"Nivel de riesgo: {resultado["Riesgo"]}")

print("\n✅ Implementación completada")

###14. CONCLUSIONES

In [ ]:
#---------------------------------CONCLUSIONES-----------------------
#Impresión por pantalla de las conclusiones finales extraídas del estudio
console.rule("[#00ffff]Conclusiones Finales[/#00ffff]", style="bright_magenta")

#Obtención de las métricas finales del modelo
y_pred_final=modelo_final.predict(x_test_scaled)
mc_final=confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp=mc_final.ravel()

#Cálculo de las métricas importantes
sensibilidad_final=recall_score(y_test, y_pred_final)
especificidad_final=tn / (tn + fp) if (tn + fp) > 0 else 0

# Mostrar conclusiones
print("\n" + "="*70)
print("RESUMEN FINAL DEL TFM")
print("="*70)

print("\n🎯 Objetivo:", "\n", "Desarrollo y evaluación del modelo de clasificación",
      "supervisada con el fin de predecir casos de diabetes en mujeres,",
      "priorizando la minimización de los falsos negativos")


print(f"\n🏆 Modelo seleccionado: {mejor_nombre}")

print("\n📊 Rendimiento del Modelo Final:")
print(f"  • Sensibilidad (Recall): {sensibilidad_final:.4f} ({sensibilidad_final*100:.1f}%)")
print(f"  • Especificidad: {especificidad_final:.4f} ({especificidad_final*100:.1f}%)")
print(f"  • Verdaderos Positivos (diabetes detectada): {tp}")
print(f"  • Falsos Negativos (diabetes NO detectada): {fn} ← Parámetro CRÍTICO")
print(f"  • Verdaderos Negativos (sanos identificados): {tn}")
print(f"  • Falsos Positivos (falsas alarmas): {fp}")

print("\n")

print(f"\n Implicaciones Clínicas:")
print(f"  • El modelo detecta correctamente {sensibilidad_final*100:.1f}% de los casos de diabetes.")
print(f"  • {fn} pacientes con diabetes no serían identificados, suponiendo un riesgo para los pacientes",
      "ya que retrasaría su diagnóstico.")
print(f"  • Esta herramienta siempre debe usarse bajo la supervisión del médico como complementaria.")



print(f"\n🔍 Limitaciones:")
print(f"  • El dataset solo presenta datos de mujeres de la etnia Pima, por lo tanto",
      "no puede ser generalizado a toda la población.")
print(f"  • Variables clínicamente relevantes como los niveles de insulina presentaban un alto porcentaje de missing values.")

print(f"\n🚀 Recomendaciones para el Futuro:")
print(f"  • Validar el modelo en otras poblaciones y centros hospitalarios.")
print(f"  • Incorporar más variables clínicas y genéticas.")
print(f"  • Implementar en entorno real para evaluar utilidad práctica.")

print("\n" + "="*70)
print("✅ TRABAJO COMPLETADO")
print("="*70)

# Guardar conclusiones
with open('conclusiones_tfm.txt', 'w', encoding='utf-8') as f:
    f.write(f"CONCLUSIONES FINALES - TFM DIABETES\n")
    f.write(f"="*50 + "\n\n")
    f.write(f"Mejor modelo: {mejor_nombre}\n")
    f.write(f"Sensibilidad: {sensibilidad_final:.4f} ({sensibilidad_final*100:.1f}%)\n")
    f.write(f"Especificidad: {especificidad_final:.4f} ({especificidad_final*100:.1f}%)\n")
    f.write(f"Falsos Negativos: {fn}\n")
    f.write(f"Falsos Positivos: {fp}\n")

print("\n✅ Conclusiones guardadas en 'conclusiones_tfm.txt'")